<a href="https://colab.research.google.com/github/Sagnik-Chowdhury/Federated-Learning-1/blob/Sourit/Fed_Gradient_Clipping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradient Clipping & Differential Privacy (DP-SGD)
### Analyzing Privacy Noise Across Data Modalities

**Objective:**
This notebook simulates an industry-standard Differential Privacy pipeline (DP-SGD) in a Federated Learning environment. We are testing how restricting client updates (Clipping) and injecting statistical noise (Laplace vs. Gaussian) impacts different data structures.

**The Hypothesis:**
Dense, highly integrated data (like MNIST pixels) relies on fragile spatial correlations that are easily destroyed by heavy-tailed privacy noise. Scattered, tabular data (like Breast Cancer features) is hypothesized to be much more robust to Differential Privacy mechanisms, specifically Laplace noise.

## Libraries

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
import copy
import pandas as pd
import math

## Architecture

In [15]:
#  For Integrated Data (MNIST Images)
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)

In [16]:
# For Scattered Data (Tabular Features)
# The Breast Cancer dataset has 30 input features and 2 output classes
class TabularNet(nn.Module):
    def __init__(self):
        super(TabularNet, self).__init__()
        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

## Data Preparation

In [17]:
NUM_CLIENTS = 20

In [18]:
print("Preparing MNIST Dataset (Integrated Data)")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

mnist_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_split = random_split(mnist_full, [len(mnist_full) // NUM_CLIENTS] * NUM_CLIENTS)
mnist_loaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in mnist_split]

mnist_test = datasets.MNIST('./data', train=False, download=True, transform=transform)
mnist_test_loader = DataLoader(mnist_test, batch_size=1000, shuffle=False)

print("\nData preparation complete")

Preparing MNIST Dataset (Integrated Data)

Data preparation complete


In [19]:
print("Preparing Breast Cancer Dataset (Scattered/Tabular Data)")

# Load and scale tabular data
data = load_breast_cancer()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data.data)
y = data.target

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
tabular_full = TensorDataset(X_tensor, y_tensor)

# Split tabular data among clients
# Tabular datasets are much smaller, so batch size is reduced
tab_split_size = len(tabular_full) // NUM_CLIENTS
# Handle remainders if dataset doesn't divide perfectly
tab_splits = [tab_split_size] * NUM_CLIENTS
tab_splits[-1] += len(tabular_full) % NUM_CLIENTS
tabular_loaders = [DataLoader(ds, batch_size=8, shuffle=True) for ds in random_split(tabular_full, tab_splits)]
tabular_test_loader = DataLoader(tabular_full, batch_size=len(tabular_full), shuffle=False)

print("\nData preparation complete")

Preparing Breast Cancer Dataset (Scattered/Tabular Data)

Data preparation complete


## The Defense Mechanism: Gradient Clipping (L2 Norm)

Unlike Trimmed Mean which discards entire clients based on quantiles, **Gradient Clipping** acts as a strict mathematical speed limit for every participating client.

**How it works:**
1. The server calculates the mathematical difference between the global model and the client's proposed update.
2. It calculates the magnitude (**L2 Norm**) of that update.
3. If the magnitude exceeds our strict `clip_threshold` (set to 2.0), the update is mathematically scaled down so it cannot overpower the global model.
4. Finally, statistical noise (Normal or Laplace) is added to the clipped average to guarantee Differential Privacy.

In [23]:
def gradient_clipping_aggregation(client_weights_list, global_weights, clip_threshold=1.0):
    """
    Calculates the update each client wants to make, measures its L2 norm,
    and scales it down if it exceeds the clip_threshold.
    """
    clipped_updates = []

    # 1. Calculate and clip each client's update
    for client_weights in client_weights_list:
        client_update = {}
        squared_sum = 0.0

        # Calculate the mathematical difference (update) the client made
        for key in global_weights.keys():
            update_tensor = client_weights[key] - global_weights[key]
            client_update[key] = update_tensor
            squared_sum += torch.sum(update_tensor ** 2).item()

        # Calculate the L2 Norm (magnitude) of the entire update
        l2_norm = math.sqrt(squared_sum)

        # Calculate the scaling factor (if norm > threshold, scale > 1)
        scale = max(1.0, l2_norm / clip_threshold)

        # Apply the speed limit (scale down if necessary)
        clipped_client_update = {}
        for key in client_update.keys():
            clipped_client_update[key] = client_update[key] / scale

        clipped_updates.append(clipped_client_update)

    # 2. Average the safely clipped updates
    avg_update = copy.deepcopy(clipped_updates[0])
    for key in avg_update.keys():
        stacked_updates = torch.stack([update[key] for update in clipped_updates])
        avg_update[key] = torch.mean(stacked_updates, dim=0)

    # 3. Apply the safe average update to the global model
    new_global_weights = copy.deepcopy(global_weights)
    for key in new_global_weights.keys():
        new_global_weights[key] += avg_update[key]

    return new_global_weights

In [27]:
def add_dp_noise(weights, noise_type='none', scale=0.01):
    """
    Injects Differential Privacy noise into the aggregated weights.
    Compares Laplace (good for sparse/scattered) vs Normal (Gaussian).
    """
    if noise_type == 'none':
        return weights

    noisy_weights = copy.deepcopy(weights)

    for key in noisy_weights.keys():
        tensor = noisy_weights[key]

        if noise_type == 'normal':
            noise = torch.randn_like(tensor) * scale
        elif noise_type == 'laplace':
            m = torch.distributions.laplace.Laplace(torch.tensor([0.0]), torch.tensor([scale]))
            noise = m.sample(tensor.shape).squeeze(-1).to(tensor.device)

        noisy_weights[key] = tensor + noise

    return noisy_weights

## Automated Grid Search Execution

The following loop executes a completely automated grid search across two data modalities (Image vs. Tabular) and three privacy states (Baseline, Gaussian Noise, and Laplace Noise).

* **Federated Rounds:** 5
* **Clients:** 20
* **DP Noise Scale:** 0.05
* **Clipping Threshold:** 2.0

In [24]:
datasets_to_test = ['mnist', 'tabular']
noise_types_to_test = ['none', 'normal', 'laplace']
NOISE_SCALE = 0.05
federated_rounds = 5
epochs_per_round = 1

experiment_results = {'mnist': {}, 'tabular': {}}

print("Starting Automated Grid Search for Gradient Clipping")

for dataset in datasets_to_test:
    for noise in noise_types_to_test:
        print(f"\nTesting {dataset.upper()} with {noise.upper()} noise")

        # 1. SETUP & RESET THE MODEL
        if dataset == 'mnist':
            global_model = MNISTNet()
            loaders = mnist_loaders
            test_loader = mnist_test_loader
            lr = 0.001
        else:
            global_model = TabularNet()
            loaders = tabular_loaders
            test_loader = tabular_test_loader
            lr = 0.01

        # 2. THE EXECUTION LOOP (Training)
        for round_num in range(federated_rounds):
            client_weights = []

            for client_idx in range(NUM_CLIENTS):
                local_model = MNISTNet() if dataset == 'mnist' else TabularNet()
                local_model.load_state_dict(global_model.state_dict())

                optimizer = optim.Adam(local_model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()

                local_model.train()
                for epoch in range(epochs_per_round):
                    for inputs, labels in loaders[client_idx]:
                        optimizer.zero_grad()
                        outputs = local_model(inputs)
                        loss = criterion(outputs, labels)
                        loss.backward()
                        optimizer.step()

                client_weights.append(local_model.state_dict())

            # --- TASK 3 MODIFICATION: GRADIENT CLIPPING ---
            aggregated_weights = gradient_clipping_aggregation(
                client_weights,
                global_model.state_dict(),
                clip_threshold=2.0
            )

            # Add the DP Noise
            secured_weights = add_dp_noise(aggregated_weights, noise_type=noise, scale=NOISE_SCALE)
            global_model.load_state_dict(secured_weights)

        # 3. THE EVALUATION (Testing)
        global_model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = global_model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        final_accuracy = 100 * correct / total
        print(f" Accuracy: {final_accuracy:.2f}% ")

        experiment_results[dataset][noise] = round(final_accuracy,3)


Starting Automated Grid Search for Gradient Clipping

Testing MNIST with NONE noise
 Accuracy: 90.51% 

Testing MNIST with NORMAL noise
 Accuracy: 68.15% 

Testing MNIST with LAPLACE noise
 Accuracy: 32.97% 

Testing TABULAR with NONE noise
 Accuracy: 97.72% 

Testing TABULAR with NORMAL noise
 Accuracy: 97.72% 

Testing TABULAR with LAPLACE noise
 Accuracy: 97.36% 


In [28]:
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

results_df = pd.DataFrame(experiment_results).T
results_df.columns = ['Baseline (No DP)', 'Normal (Gaussian)', 'Laplace']
results_df.index = ['MNIST (Dense)', 'Breast Cancer (Scattered)']

print(results_df.to_string())
print("="*50)


FINAL RESULTS
                           Baseline (No DP)  Normal (Gaussian)  Laplace
MNIST (Dense)                        90.510             68.150   32.970
Breast Cancer (Scattered)            97.715             97.715   97.364


## Final Observations & Conclusion

The experiment yielded dramatic, conclusive results regarding data modality and Differential Privacy:

1. **Integrated Data is Fragile:** The MNIST model suffered catastrophic forgetting when exposed to heavy-tailed noise, dropping from a baseline of **90.51%** down to **68.15%** (Gaussian) and plummeting to **32.97%** (Laplace).

2. **Scattered Data is Robust:** The Breast Cancer tabular model showed incredible resilience. It maintained a **97.71%** accuracy under Gaussian noise and a highly stable **97.36%** under the heavier Laplace noise.

**Conclusion:** The structure of the data dictates the defense. Federated models handling scattered tabular features can easily survive aggressive, heavy-tailed privacy mechanisms like Laplace distributions. However, applying that same mathematical noise to dense image data destroys the learned spatial patterns.